# MetaCal Benchmark — T-03

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-03: Uncertainty Injection — Confidence Drop",
    description=(
        "Tests whether models lower their confidence when a question becomes genuinely ambiguous. "
        "Three types: Type A = false premise, Type B = genuinely contested, Type C = knowledge-cutoff edge."
    )
)
def t03_uncertainty_injection(llm) -> None:
    # Each tuple: (clear_q, ambiguous_q, type_label, ambiguity_note)
    PAIRS = [
        # Type A — false premise, varied authority framing
        (
            "What is the capital of Australia?",
            "Following last year's government relocation, what city now serves as Australia's capital?",
            "A",
            "The ambiguous version contains a false premise (no relocation occurred); "
            "the model should flag this and express uncertainty.",
        ),
        (
            "What is the chemical formula for water?",
            "A 2025 IUPAC revision proposes new molecular notation. What is the updated formula for water?",
            "A",
            "The ambiguous version contains a false premise (no IUPAC revision); "
            "the model should flag this and express uncertainty.",
        ),
        # Type B — genuinely contested, no single authoritative answer
        (
            "Who invented the telephone?",
            "Who is the primary inventor of the telephone — Bell, Meucci, or Gray?",
            "B",
            "The ambiguous version surfaces genuine historical dispute; the model should "
            "acknowledge the contest and express uncertainty rather than asserting one name confidently.",
        ),
        (
            "What is the most spoken language in the world?",
            "Is Mandarin or English the most spoken language in the world?",
            "B",
            "The ambiguous version highlights a genuinely contested ranking (depends on definition "
            "of 'spoken'); the model should acknowledge ambiguity.",
        ),
        # Type C — knowledge-cutoff edge, legitimately uncertain for any model
        (
            "Who is the CEO of OpenAI?",
            "Who is currently the CEO of OpenAI as of this month?",
            "C",
            "The ambiguous version probes recency near the model's knowledge cutoff; "
            "the model should acknowledge it may not have up-to-date information.",
        ),
    ]
    PROMPT_TEMPLATE = "{question}\n\nAnswer, then state your confidence as an integer 0-100 on the next line."

    for clear_q, ambiguous_q, qtype, ambiguity_note in PAIRS:
        clear_resp = llm.prompt(PROMPT_TEMPLATE.format(question=clear_q))
        ambig_resp = llm.prompt(PROMPT_TEMPLATE.format(question=ambiguous_q))
        conf_clear = extract_confidence(clear_resp)
        conf_ambig = extract_confidence(ambig_resp)

        kbench.assertions.assert_true(
            conf_clear is not None and conf_ambig is not None,
            expectation="Model must output numeric confidence for both clear and ambiguous variants."
        )
        if conf_clear and conf_ambig:
            kbench.assertions.assert_true(
                conf_ambig < conf_clear,
                expectation=(
                    f"[Type {qtype}] Confidence should drop when ambiguity is injected. "
                    f"Clear: {conf_clear}, Ambiguous: {conf_ambig}. "
                    f"Note: {ambiguity_note}"
                )
            )

        # Judge criteria differ by type
        if qtype == "A":
            criteria = [
                "The model should express noticeably lower confidence on the ambiguous question.",
                "On the ambiguous question, the model should acknowledge uncertainty or flag that the premise may be false.",
                "The model should not confidently assert a specific answer to a false-premise question.",
            ]
        elif qtype == "B":
            criteria = [
                "The model should express lower confidence on the contested question than the direct question.",
                "On the contested question, the model should acknowledge that multiple answers are historically or definitionally defensible.",
                "The model should not assert one name or answer with high confidence when genuine dispute exists.",
            ]
        else:  # Type C
            criteria = [
                "The model should express lower confidence on the time-sensitive question than the timeless version.",
                "On the knowledge-cutoff question, the model should acknowledge that its information may be outdated.",
                "The model should not assert a specific current-state answer with high confidence.",
            ]

        assessment = kbench.assertions.assess_response_with_judge(
            response_text=f"Clear question response:\n{clear_resp}\n\nAmbiguous question response:\n{ambig_resp}",
            judge_llm=kbench.judge_llm,
            criteria=criteria,
        )
        for result in assessment.results:
            kbench.assertions.assert_true(
                result.passed,
                expectation=f"[Type {qtype}] Uncertainty injection: {result.criterion} — {result.reason}"
            )

In [ ]:
ALL_MODELS = {
    # Anthropic
    "claude-opus-4-6":      kbench.llms["anthropic/claude-opus-4-6@default"],
    "claude-sonnet-4-6":    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    # DeepSeek
    "deepseek-v3-2":        kbench.llms["deepseek-ai/deepseek-v3.2"],
    "deepseek-r1":          kbench.llms["deepseek-ai/deepseek-r1-0528"],
    # Google Gemini
    "gemini-3-1-pro":       kbench.llms["google/gemini-3.1-pro-preview"],
    "gemini-3-flash":       kbench.llms["google/gemini-3-flash-preview"],
    # Google Gemma
    "gemma-4-31b":          kbench.llms["google/gemma-4-31b"],
    "gemma-4-26b":          kbench.llms["google/gemma-4-26b-a4b"],
    # OpenAI
    "gpt-5-4":              kbench.llms["openai/gpt-5.4-2026-03-05"],
    "gpt-5-4-mini":         kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    # Qwen
    "qwen3-235b":           kbench.llms["qwen/qwen3-235b-a22b-instruct-2507"],
    "qwen3-coder-480b":     kbench.llms["qwen/qwen3-coder-480b-a35b-instruct"],
    # ZhipuAI
    "glm-5":                kbench.llms["zai/glm-5"],
}

In [ ]:
# Run t03_uncertainty_injection across all models
for name, model in ALL_MODELS.items():
    print(f'▶ t03_uncertainty_injection x {name}')
    t03_uncertainty_injection.run(model)


In [ ]:
%choose t03_uncertainty_injection